# Notebook 5: Model Training

## Objective
Train and compare multiple machine learning models to predict Formula 1 lap times using the engineered features.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

import joblib

In [10]:
# loading the processed dataset

master_df = pd.read_csv("../data/processed/master_dataset.csv")
print(master_df.shape)
master_df.head()

(11880, 14)


,Driver,Team,LapNumber,Compound,TyreLife,Stint,Position,TrackStatus,LapTimeSeconds,FuelLoadApprox,StintProgress,CompoundEncoded,FreshTyre,TyreAgeSquared
0,VER,Red Bull Racing,1.0,SOFT,4.0,1.0,2.0,1,83.186,0.984848,0.100,0,0,16.0
1,VER,Red Bull Racing,2.0,SOFT,5.0,1.0,2.0,1,79.871,0.969697,0.125,0,0,25.0
2,VER,Red Bull Racing,3.0,SOFT,6.0,1.0,1.0,1,79.364,0.954545,0.150,0,0,36.0
3,VER,Red Bull Racing,4.0,SOFT,7.0,1.0,1.0,1,80.766,0.939394,0.175,0,0,49.0
4,VER,Red Bull Racing,5.0,SOFT,8.0,1.0,1.0,1,80.827,0.924242,0.200,0,0,64.0


In [11]:
# features and target

features = [
    "LapNumber",
    "TyreLife",
    "Stint",
    "Position",
    "TrackStatus",
    "FuelLoadApprox",
    "StintProgress",
    "CompoundEncoded",
    "FreshTyre",
    "TyreAgeSquared"
]

X = master_df[features]
y = master_df["LapTimeSeconds"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11880, 10)
y shape: (11880,)


In [12]:
# splitting into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 9504
Testing samples: 2376


## Linear regression


In [13]:
# Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_predictions = lr.predict(X_test)

In [14]:
# Evaluation

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", round(lr_mae, 3))
print("RMSE:", round(lr_rmse, 3))
print("R²  :", round(lr_r2, 3))

Linear Regression Results
-------------------------
MAE : 7.727
RMSE: 8.801
R²  : 0.099


## Random Forest

In [15]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_predictions = rf.predict(X_test)

In [16]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("---------------------")
print("MAE :", round(rf_mae, 3))
print("RMSE:", round(rf_rmse, 3))
print("R²  :", round(rf_r2, 3))

Random Forest Results
---------------------
MAE : 5.378
RMSE: 7.086
R²  : 0.416


## XGBoost

In [17]:
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_predictions = xgb.predict(X_test)

In [18]:
xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

print("XGBoost Results")
print("----------------")
print("MAE :", round(xgb_mae, 3))
print("RMSE:", round(xgb_rmse, 3))
print("R²  :", round(xgb_r2, 3))

XGBoost Results
----------------
MAE : 6.11
RMSE: 7.27
R²  : 0.385


In [19]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2": [
        lr_r2,
        rf_r2,
        xgb_r2
    ]
})

results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
1,Random Forest,5.378498,7.086379,0.415954
2,XGBoost,6.109555,7.269950,0.385303
0,Linear Regression,7.726958,8.801269,0.099074


In [20]:
# since random forest performed well, we will save it

import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(rf, "../models/best_model.pkl")

print("Best model saved successfully!")

Best model saved successfully!
